# 01. Optical flow, warping과 cycle consistency

목표: 1차원 signal을 이용해 flow, warping, photometric error, EPE와 forward-backward cycle consistency를 익힌다. 실제 2차원 differentiable warping이 아닌 개념 실습이다.

In [ ]:
def warp_1d(signal, integer_flow):
    """각 원소를 integer_flow만큼 이동하고 경계 밖 값은 버린다."""
    warped = [0.0] * len(signal)
    valid = [False] * len(signal)
    for source_index, value in enumerate(signal):
        target_index = source_index + integer_flow
        if 0 <= target_index < len(signal):
            warped[target_index] = value
            valid[target_index] = True
    return warped, valid

frame_t = [0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
frame_next, valid = warp_1d(frame_t, integer_flow=1)
print("frame t:  ", frame_t)
print("frame t+1:", frame_next)
print("valid:    ", valid)

In [ ]:
def masked_mse(prediction, target, valid):
    errors = [
        (predicted - expected) ** 2
        for predicted, expected, keep in zip(prediction, target, valid)
        if keep
    ]
    return sum(errors) / len(errors)

for predicted_flow in (0, 1, 2):
    prediction, prediction_valid = warp_1d(frame_t, predicted_flow)
    print(
        f"flow={predicted_flow}, "
        f"photometric MSE={masked_mse(prediction, frame_next, prediction_valid):.3f}"
    )

In [ ]:
def end_point_error(predicted_flow, true_flow):
    # 1차원에서는 vector L2가 절댓값과 같다.
    return abs(predicted_flow - true_flow)

def cycle_error(forward_flow, backward_flow):
    # 완전한 대응 영역에서는 왕복 이동의 합이 0이어야 한다.
    return abs(forward_flow + backward_flow)

print("EPE:", end_point_error(predicted_flow=0.8, true_flow=1.0))
print("올바른 cycle:", cycle_error(1.0, -1.0))
print("불일치 cycle:", cycle_error(1.0, -0.6))

## 실습 과제

1. 밝기가 변한 다음 frame을 만들고 photometric consistency의 한계를 확인한다.
2. 여러 위치에 서로 다른 flow를 주는 vector field로 확장한다.
3. 경계 밖으로 이동한 원소를 loss에서 제외해야 하는 이유를 occlusion과 연결한다.
4. 2차원 bilinear sampling에서 integer shift와 달라지는 점을 설명한다.